In [1]:
import pandas as pd 

In [3]:
df=pd.read_csv("C:/Users/GOWRI/Downloads/INTERNSHIP PROJECT.csv/final2_NYC.csv")
df


,vendor_id,passenger_count,pickup_daytype,pickup_day,pickup_hour,distance,trip_duration
0,2,1,0,0,17,0.019859,455
1,1,1,1,6,0,0.026478,663
2,2,1,0,1,11,0.080158,2124
3,2,1,0,2,19,0.015480,429
4,2,1,1,5,13,0.010818,435
...,...,...,...,...,...,...,...
1458579,2,4,0,4,13,0.018063,778
1458580,1,1,1,6,7,0.079929,655
1458581,2,1,0,4,6,0.106731,764
1458582,1,1,0,1,15,0.015491,373


In [21]:
print("Rows BEFORE cleaning:", len(df))

df = df[
    (df['trip_duration'].between(60, 7200)) &  # 1 min to 2 hours
    (df['passenger_count'].between(1, 6))       # valid passenger count
]

print("Rows AFTER  cleaning:", len(df))
print(f"Removed: {1458584 - len(df):,} outlier rows")


Rows BEFORE cleaning: 1458584
Rows AFTER  cleaning: 1447777
Removed: 10,807 outlier rows


In [47]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

In [22]:
X = df.drop('trip_duration',axis=1)
y = df['trip_duration']

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,     # 80% train, 20% test
    random_state=42
)

print(f"Training samples : {len(X_train):,}")
print(f"Testing  samples : {len(X_test):,}")

Training samples : 1,158,221
Testing  samples : 289,556


In [25]:
def evaluate(y_true, y_pred, model_name):
    """
    Evaluate model performance directly in seconds/minutes.
    No log/expm1 needed anymore!
    """
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))

    print(f"\n{'='*45}")
    print(f"  {model_name}")
    print(f"{'='*45}")
    print(f"  R² Score   : {r2:.4f}   (closer to 1.0 = better)")
    print(f"  MAE        : {mae:.1f} sec  = {mae/60:.1f} min avg error")
    print(f"  RMSE       : {rmse:.1f} sec  = {rmse/60:.1f} min")

    return dict(R2=round(r2,4), MAE_s=round(mae,1), RMSE_s=round(rmse,1))



In [27]:
# Ridge still needs scaling because it's a linear model
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
yp_ridge = ridge.predict(X_test_scaled)

results_ridge = evaluate(y_test, yp_ridge, "Ridge Regression")



  Ridge Regression
  R² Score   : 0.3493   (closer to 1.0 = better)
  MAE        : 299.2 sec  = 5.0 min avg error
  RMSE       : 527.6 sec  = 8.8 min


In [28]:
# No scaling needed — Random Forest is tree-based
# No log1p needed — predicting raw trip_duration directly

np.random.seed(42)
sample_idx = np.random.choice(len(X_train), 300_000, replace=False)

rf = RandomForestRegressor(
    n_estimators=100,   # 100 decision trees
    max_depth=12,       # max depth of each tree
    n_jobs=-1,          # use all CPU cores
    random_state=42
)

rf.fit(X_train.iloc[sample_idx], y_train.iloc[sample_idx])
yp_rf = rf.predict(X_test)

results_rf = evaluate(y_test, yp_rf, "Random Forest")

# Feature importance — which input mattered most?
print("\n--- Feature Importance ---")
FEATURES = X.columns
importance = pd.Series(rf.feature_importances_, index=FEATURES)
importance = importance.sort_values(ascending=False)
for feat, score in importance.items():
    bar = '█' * int(score * 100)
    print(f"  {feat:<20} {bar}  {score:.4f}")


  Random Forest
  R² Score   : 0.7018   (closer to 1.0 = better)
  MAE        : 238.9 sec  = 4.0 min avg error
  RMSE       : 357.2 sec  = 6.0 min

--- Feature Importance ---
  distance             ███████████████████████████████████████████████████████████████████████████████████████  0.8738
  pickup_hour          ████████  0.0860
  pickup_day           ██  0.0228
  pickup_daytype       █  0.0117
  passenger_count        0.0041
  vendor_id              0.0017


In [29]:
# No scaling needed — also tree-based
# No log1p needed

np.random.seed(0)
sample_idx2 = np.random.choice(len(X_train), 150_000, replace=False)

gb = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)

gb.fit(X_train.iloc[sample_idx2], y_train.iloc[sample_idx2])
yp_gb = gb.predict(X_test)

results_gb = evaluate(y_test, yp_gb, "Gradient Boosting")


  Gradient Boosting
  R² Score   : 0.7030   (closer to 1.0 = better)
  MAE        : 238.5 sec  = 4.0 min avg error
  RMSE       : 356.4 sec  = 5.9 min


In [30]:
print("\n" + "=" * 55)
print(f"{'Model':<22} {'R²':>6} {'MAE(min)':>10} {'RMSE(min)':>10}")
print("=" * 55)

all_results = {
    'Ridge Regression':  results_ridge,
    'Random Forest':     results_rf,
    'Gradient Boosting': results_gb,
}

for name, r in all_results.items():
    print(f"  {name:<22} {r['R2']:>6.4f} "
          f"{r['MAE_s']/60:>10.1f} {r['RMSE_s']/60:>10.1f}")

print("=" * 55)

best_name = max(all_results, key=lambda k: all_results[k]['R2'])
print(f"\n✅ Best Model : {best_name}")
print(f"   R²        : {all_results[best_name]['R2']}")
print(f"   Avg Error : {all_results[best_name]['MAE_s']/60:.1f} minutes")


Model                      R²   MAE(min)  RMSE(min)
  Ridge Regression       0.3493        5.0        8.8
  Random Forest          0.7018        4.0        6.0
  Gradient Boosting      0.7030        4.0        5.9

✅ Best Model : Gradient Boosting
   R²        : 0.703
   Avg Error : 4.0 minutes


In [48]:
from xgboost import XGBRegressor

# No scaling needed — tree-based
# No log1p needed

np.random.seed(1)
sample_idx3 = np.random.choice(len(X_train), 150_000, replace=False)

xgb = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42
)

xgb.fit(X_train.iloc[sample_idx3], y_train.iloc[sample_idx3])
yp_xgb = xgb.predict(X_test)

results_xgb = evaluate(y_test, yp_xgb, "XGBoost")


  XGBoost
  R² Score   : 0.7036   (closer to 1.0 = better)
  MAE        : 238.6 sec  = 4.0 min avg error
  RMSE       : 356.1 sec  = 5.9 min


In [49]:
from catboost import CatBoostRegressor

# No scaling needed
# No log1p needed

np.random.seed(2)
sample_idx4 = np.random.choice(len(X_train), 150_000, replace=False)

cat = CatBoostRegressor(
    iterations=200,
    depth=6,
    learning_rate=0.1,
    verbose=0,
    random_state=42
)

cat.fit(X_train.iloc[sample_idx4], y_train.iloc[sample_idx4])
yp_cat = cat.predict(X_test)

results_cat = evaluate(y_test, yp_cat, "CatBoost")


  CatBoost
  R² Score   : 0.7045   (closer to 1.0 = better)
  MAE        : 238.0 sec  = 4.0 min avg error
  RMSE       : 355.5 sec  = 5.9 min


In [50]:
from sklearn.ensemble import AdaBoostRegressor

# No scaling needed
# No log1p needed

np.random.seed(3)
sample_idx5 = np.random.choice(len(X_train), 150_000, replace=False)

ada = AdaBoostRegressor(
    n_estimators=150,
    learning_rate=0.1,
    random_state=42
)

ada.fit(X_train.iloc[sample_idx5], y_train.iloc[sample_idx5])
yp_ada = ada.predict(X_test)

results_ada = evaluate(y_test, yp_ada, "AdaBoost")


  AdaBoost
  R² Score   : 0.4325   (closer to 1.0 = better)
  MAE        : 386.3 sec  = 6.4 min avg error
  RMSE       : 492.7 sec  = 8.2 min


In [51]:
print("\n" + "=" * 55)
print(f"{'Model':<22} {'R²':>6} {'MAE(min)':>10} {'RMSE(min)':>10}")
print("=" * 55)

all_results = {
    'Ridge Regression':  results_ridge,
    'Random Forest':     results_rf,
    'Gradient Boosting': results_gb,
    'XGBoost':           results_xgb,
    'CatBoost':          results_cat,
    'AdaBoost':          results_ada,
}

for name, r in all_results.items():
    print(f"  {name:<22} {r['R2']:>6.4f} "
          f"{r['MAE_s']/60:>10.1f} {r['RMSE_s']/60:>10.1f}")

print("=" * 55)

best_name = max(all_results, key=lambda k: all_results[k]['R2'])
print(f"\n✅ Best Model : {best_name}")
print(f"   R²        : {all_results[best_name]['R2']}")
print(f"   Avg Error : {all_results[best_name]['MAE_s']/60:.1f} minutes")


Model                      R²   MAE(min)  RMSE(min)
  Ridge Regression       0.3493        5.0        8.8
  Random Forest          0.7018        4.0        6.0
  Gradient Boosting      0.7030        4.0        5.9
  XGBoost                0.7036        4.0        5.9
  CatBoost               0.7045        4.0        5.9
  AdaBoost               0.4325        6.4        8.2

✅ Best Model : CatBoost
   R²        : 0.7045
   Avg Error : 4.0 minutes
